# Batch image processing with multimodal API

## contact hongyuan.zhang@usys.ethz.ch

(Optional) if you want to connect with your Google Driver

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

%cd /content/drive/MyDrive/

## 1. Clone the code and store your API keys in the secrets
name it as `api_key`

In [1]:
!git clone https://github.com/Alias-z/FungariumOCR.git -b llm_course
%cd FungariumOCR

Cloning into 'FungariumOCR'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 60 (delta 9), reused 26 (delta 7), pack-reused 30 (from 1)
Receiving objects: 100% (60/60), 43.25 MiB | 18.08 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/FungariumOCR


## 2. Define your prompts and structured output

In [5]:
#@title **Adjust prompts settings and output strcture**

from pydantic import BaseModel

example_case = 'cat_breed' #@param ['fungi_ocr', 'cat_breed']

if example_case == 'fungi_ocr':

  system_prompt = """
          The goal is to extract structured text from Fungi sample images. The rules are:
          1. Each image contains two sections of text chunks. One is the barcode, and the other is the sample information.

          2. The languages are only in English or German.

          3. The bar code chunk starts with 'Herbarium der ETH Zurich (ZT)' followed by a barcode with text such as 'ZT Myc 0105537'. The task here is to extract the barcode text.

          4. The sample information chunk starts with a division separator, such as 'Dr. F. Petrak, Mycotheca generalis.'
          This will become the value for the 'division' column. After the division separator, there are other structures defined by the following:

          5. Exicata Number and Species: Lines. For example, '204. Acetabula vulgaris Fuck', contains two pieces of information: * Exicata number → the number before the period (e.g., 204) * Species name → everything after the period (e.g., Acetabula vulgaris Fuck.)

          6. Matrix and Locality: A line that holds the information (e.g., Ungarn; Comit. Gyor: Bonyretalap). If a sample is missing a Matrix and Locality line, leave it blank. Extract as it is, no more added information

          7. Date: A line that has a Roman numeral month plus year (e.g., V.1920, X.1924, XII.1924), from which you split out: * Month → Roman numeral (e.g., V, X, XII) * Year → numeric year (e.g., 1920, 1924). Note, sometimes the month can be a normal English month with abbreviations

          8. Collector: A line beginning with 'leg.' indicates the collector (e.g., leg. J. Cogolludo.)

          9. Image name: I will define later

          The output should be a JSON like structured dictionary with keys (image_name, barcode, division, exicata_number, species, matrix_locality, date, collector)
          Remove unnecessary '\n' etc. Dont output anything else.
          """

  user_prompt = 'Directly extract information with your own vision capabilities, not Python packages such as pytesseract'

  input_dir = 'sample_images'

  class OutputFormat(BaseModel):
      image_name: str
      barcode: str
      division: str
      exicata_number: str
      species: str
      matrix_locality: str
      date: str
      collector: str



elif example_case == 'cat_breed':

  system_prompt = """
          The goal is to get the cat breed from the image. The rules are:

          The output should be a JSON like structured dictionary with keys (image_name, cat_breed)

          Remove unnecessary '\n' etc. Dont output anything else.
          """

  user_prompt = 'Directly extract information with your own vision capabilities, not Python packages'


  input_dir = 'sample_image_cats'


  class OutputFormat(BaseModel):
      image_name: str
      cat_breed: str

In [6]:
#@title **Adjust model settings**
from google.colab import userdata
from pydantic import BaseModel
from fungarium_ocr.core import FungariumOCR

#@markdown  **Mandatory parameters**

api_source = 'sph_ethz' #@param ['openai', 'sph_ethz']
vsion_model = 'gpt-4o' #@param ['gpt-4o', 'gpt-4o-mini']
# input_dir = 'sample_images' #@param {type: 'string'}

#@markdown  **Optional (Do not modify if you are not sure)**

model_temperature = 0.5 #@param {type:'slider', min:0, max:1, step:0.1}
image_resize_ratio = 0.5 #@param {type:'slider', min:0, max:1, step:0.1}

processor = FungariumOCR(openai_apikey=userdata.get('api_key'), api_source=api_source)

processor.batch_ocr(input_dir,
                    vsion_model=vsion_model,
                    system_prompt=system_prompt,
                    user_prompt=user_prompt,
                    response_format=OutputFormat,
                    temperature=model_temperature,
                    resize_ratio=image_resize_ratio)

Processing images: 100%|██████████| 2/2 [00:04<00:00,  2.25s/it]

Successfully saved Excel file to: sample_image_cats/sample_image_cats.xlsx
